In [39]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder

In [40]:
df = pd.read_csv("src/data/inference/raw/merged_inference_data.csv")

In [41]:
df['sd__membership_product'].unique()

array(['BUSINESS_SMART', 'PERSONAL_STANDARD', 'PERSONAL_METAL',
       'BUSINESS_STANDARD', 'PERSONAL_SMART', 'BUSINESS_YOU',
       'PERSONAL_YOU', 'BUSINESS_METAL', 'PERSONAL_FLEX', nan],
      dtype=object)

In [42]:
user_ids = df['user_id']

In [43]:
df["mix__sd_od_history_to_kyc_ratio"] = (
    df["od__history_length_days"] /
    df["sd__age_since_first_kyc"].replace(0, np.nan)
).fillna(0.0)

In [44]:
ordinal_features = [
    "cb__worst_score",
    "sd__membership_product"
]

ordinal_encoder = OrdinalEncoder(
    categories=[
        # example – replace with your actual order
        ["A", "B", "C", "D", "E", "F", "G", "M", "N", "O", "P", np.nan],           # cb__worst_score
        ["PERSONAL_FLEX", "PERSONAL_STANDARD", "PERSONAL_SMART"
         , "PERSONAL_YOU", "PERSONAL_METAL", "BUSINESS_STANDARD"
         , "BUSINESS_SMART", "BUSINESS_YOU", "BUSINESS_METAL", np.nan]        # sd__membership_product
    ]
)

df[ordinal_features] = ordinal_encoder.fit_transform(
    df[ordinal_features]
)

In [45]:
best_clf_model = xgb.XGBClassifier()
best_clf_model.load_model("xgb_clf_ccf.json")

best_reg_model = xgb.XGBRegressor()
best_reg_model.load_model("xgb_reg_ccf.json")

In [46]:
clf_features = best_clf_model.get_booster().feature_names
reg_features = best_reg_model.get_booster().feature_names

X_clf = df[clf_features]
X_reg = df[reg_features]

In [47]:
threshold = 0.223114

def predict_ccf(X_clf, X_reg, best_clf_model, best_reg_model, threshold=0.223114, upper_clipping_limit=1.0, bottom_clipping_limit=0.0):
    """
    Applies the Hurdle Model logic to predict CCF for new data.
    """
    # 1. Classification (The Hurdle)
    clf_probabilities = best_clf_model.predict_proba(X_clf)[:, 1]
    predicted_ccf = np.zeros(X_clf.shape[0])
    non_zero_indices = clf_probabilities >= threshold
    
    if np.any(non_zero_indices):
        # 2. Regression (The Value)
        X_reg_input = X_reg[non_zero_indices]
        reg_predictions = best_reg_model.predict(X_reg_input)
        
        # 3. Clipping (The Constraint)
        final_values = np.clip(reg_predictions, a_min=bottom_clipping_limit, a_max=upper_clipping_limit)
        
        # 4. Final Output Assembly
        predicted_ccf[non_zero_indices] = final_values
        
    return predicted_ccf

In [48]:
THRESHOLD = 0.223114  # example – use your calibrated value

df["ccf"] = predict_ccf(
    X_clf=X_clf,
    X_reg=X_reg,
    best_clf_model=best_clf_model,
    best_reg_model=best_reg_model,
    threshold=THRESHOLD
)

In [49]:
y_pred_clf = best_clf_model.predict_proba(X_clf)[:,1]
y_pred_reg = best_reg_model.predict(X_reg)

df["ccf"] = y_pred_clf*np.clip(a=y_pred_reg, a_min=0.0, a_max=1)

In [50]:
df

,user_id,reference_date,ob__limit_ref,ob__balance_ref,ob__open_limit_ref,ob__avg_util_ref,mv__is_lisbon_v1,mv__is_lisbon_v2,mv__is_lisbon_v3,mv__is_lisbon_v4,...,ab__inflow_spike_count_6m,ab__days_down_6m,ab__near_zero_days_6m,dn__recent_action,dn__max_action_level,cb__most_recent_score,cb__worst_score,ep__cc_tbil_balance,mix__sd_od_history_to_kyc_ratio,ccf
0,ebc4f5eb-ae1f-4386-a9dc-a3197aa0046a,2025-12-31,500.0,460.67,0.078660,0.921340,0,0,0,0,...,1,37,0,NaN,NaN,C,2.0,0.00,0.081128,0.523129
1,ebf069af-c1b7-44b3-8e8e-bf40a2e0a94e,2025-12-31,500.0,0.00,1.000000,0.000000,1,0,0,0,...,3,13,0,NaN,NaN,A,1.0,0.00,1.000369,0.177030
2,ebf9052e-aa04-45ea-b058-7b574f637dbc,2025-12-31,10000.0,0.00,1.000000,0.000000,0,0,0,1,...,20,150,0,NaN,NaN,B,1.0,0.00,0.459339,0.122243
3,ec0af744-bcfe-4f50-82b1-4f58b8a5c7f7,2025-12-31,10000.0,0.00,1.000000,0.000000,1,0,0,0,...,10,29,0,NaN,NaN,A,0.0,0.00,1.000355,0.038054
4,ec195022-ed70-4697-833a-c7c63859f951,2025-12-31,1000.0,0.00,1.000000,0.000000,1,0,0,0,...,0,0,181,NaN,NaN,B,1.0,0.00,1.000351,0.036834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164751,b474a26e-2687-4c7f-bf19-a659d57870c5,2025-12-31,500.0,0.00,1.000000,0.000000,1,0,0,0,...,16,114,2,NaN,NaN,D,3.0,0.00,0.755887,0.195734
164752,a800ea6e-0b3d-49bf-b012-a6bc8f58fccb,2025-12-31,500.0,11.16,0.977680,0.022320,1,0,0,0,...,0,2,181,NaN,NaN,A,3.0,0.00,0.997774,0.019270
164753,a837d2a1-c5a9-4196-8adc-6066510da033,2025-12-31,6000.0,741.56,0.876406,0.123593,1,0,0,0,...,4,22,23,NaN,NaN,B,7.0,4677.79,0.772414,0.442593
164754,a866d150-5aae-4699-afed-5d2ebceb8007,2025-12-31,500.0,506.63,0.000000,1.013260,0,0,0,0,...,6,39,25,NaN,NaN,E,7.0,0.00,0.186120,0.256133


In [51]:
final_df = df[['user_id', 'reference_date', 'ccf']]

In [52]:
final_df.to_csv("ccf__output_31.12.2025_v2.csv", index=False)

In [53]:
df['ccf'].mean()

np.float32(0.29997522)

In [54]:
final_df

,user_id,reference_date,ccf
0,ebc4f5eb-ae1f-4386-a9dc-a3197aa0046a,2025-12-31,0.523129
1,ebf069af-c1b7-44b3-8e8e-bf40a2e0a94e,2025-12-31,0.177030
2,ebf9052e-aa04-45ea-b058-7b574f637dbc,2025-12-31,0.122243
3,ec0af744-bcfe-4f50-82b1-4f58b8a5c7f7,2025-12-31,0.038054
4,ec195022-ed70-4697-833a-c7c63859f951,2025-12-31,0.036834
...,...,...,...
164751,b474a26e-2687-4c7f-bf19-a659d57870c5,2025-12-31,0.195734
164752,a800ea6e-0b3d-49bf-b012-a6bc8f58fccb,2025-12-31,0.019270
164753,a837d2a1-c5a9-4196-8adc-6066510da033,2025-12-31,0.442593
164754,a866d150-5aae-4699-afed-5d2ebceb8007,2025-12-31,0.256133
